In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
master_enron = pd.read_csv('../data/master_enron.csv', low_memory=False)

## Feature Engineering
>
> Below we are going to add additional features to better understand out target which is prompt email responses. Here we are going to break down each individual email send into different columns that define email length, word complexity and the amount of filler words.

In [9]:
# Fill any missing email bodies with an empty string to prevent errors
master_enron['EmailSend'] = master_enron['EmailSend'].fillna('')

# Count the tokens by splitting on whitespace and measuring the length
master_enron['EmailSend_Token_Count'] = master_enron['EmailSend'].str.split().str.len()

# 3. Preview the text alongside your brand-new token count column
master_enron[['EmailSend', 'EmailSend_Token_Count']].head()

,EmailSend,EmailSend_Token_Count
0,"Nikki, Thanks for the note and I hope your doi...",504
1,Jim Lokay Sales Representative British Parts I...,11
2,Congratulations! Have a good sleep.,5
3,Test,1
4,We are dropping a lot of marketers. It would b...,16


In [11]:
!pip install syllables

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 992.4 kB/s eta 0:00:00a 0:00:01
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 8.5.0
    Uninstalling importlib_metadata-8.5.0:
      Successfully uninstalled importlib_metadata-8.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [syllables]


In [12]:
import syllables
import re

def fast_email_syllables(text):
    if pd.isna(text):
        return 0
    # Clean out special characters and split into individual words
    words = re.findall(r'\b\w+\b', str(text).lower())
    return sum(syllables.estimate(w) for w in words)

# Apply to your column (replace 'email_column' with your actual column name)
master_enron['syllable_count'] = master_enron['EmailSend'].apply(fast_email_syllables)

In [ ]:
def average_word_length(text):
    if pd.isna(text) or not str(text).strip():
        return 0
    
    # Extract only the alphanumeric words
    words = re.findall(r'\b\w+\b', str(text))
    
    if not words:
        return 0
        
    # Total characters in all words divided by the number of words
    total_characters = sum(len(word) for word in words)
    return total_characters / len(words)

# Apply it to your DataFrame
master_enron['avg_word_length'] = master_enron['EmailSend'].apply(average_word_length)

# Preview your new column
master_enron[['EmailSend', 'avg_word_length']].head()

,EmailSend,avg_word_length
0,"Nikki, Thanks for the note and I hope your doi...",3.920078
1,Jim Lokay Sales Representative British Parts I...,5.538462
2,Congratulations! Have a good sleep.,5.800000
3,Test,4.000000
4,We are dropping a lot of marketers. It would b...,3.187500


In [14]:
master_enron

,Unnamed: 0,EmailSend,EmailReply,SubjectSend,SubjectReply,From,To,DateSend,DateReply,Context,Response_Within_30_Mins,HourSent,DaySent,IsWeekend,Responded,EmailSend_Token_Count,syllable_count,avg_word_length
0,0,"Nikki, Thanks for the note and I hope your doi...",Hello hello! I was so worried that I had said ...,Portland,Re: Portland,bill.williams@enron.com,-nikole@excite.com,2001-06-18 22:44:00,2001-06-19 15:49:00,NaN,0,22.0,0.0,0,1,504,717,3.920078
1,1,Jim Lokay Sales Representative British Parts I...,HI,Call when you receive this (no rush),RE: Call when you receive this (no rush),548@britishparts.com,michelle.lokay@enron.com,2002-03-19 08:30:00,2002-03-19 08:34:00,NaN,1,8.0,1.0,0,1,11,24,5.538462
2,2,Congratulations! Have a good sleep.,Just woke up...thnx for your note. I believe t...,Thanks and,Re: Thanks and,louise.kitchen@enron.com,8774754543@skytel.com,2002-01-15 12:30:00,2002-01-15 16:29:00,NaN,0,12.0,1.0,0,1,5,9,5.800000
3,3,Test,Call back : Geir.Solberg@enron.com|Test|Test,Test,Re: Test,geir.solberg@enron.com,8776519147@skytel.com,2002-01-19 10:55:00,2002-01-19 10:56:00,NaN,1,10.0,5.0,1,1,1,1,4.000000
4,4,We are dropping a lot of marketers. It would b...,As shankman would say 'working ya',Marketers,RE: Marketers,8777865122@skytel.com,louise.kitchen@enron.com,2002-01-21 15:30:00,2002-01-21 15:44:00,NaN,1,15.0,0.0,0,1,16,20,3.187500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
359746,359746,"-----Original Message----- From: Reeves, Kathy...",NaN,FW: Positions without Sundance,NaN,john.zufferli@enron.com,lon.draper@enron.com,2001-12-05 14:22:58,NaN,NaN,0,14.0,2.0,0,0,19,38,5.300000
359747,359747,"-----Original Message----- From: ""Lerner, Larr...",NaN,FW: Hello,NaN,john.zufferli@enron.com,jzufferli@shaw.ca,2001-11-29 19:41:51,NaN,NaN,0,19.0,3.0,0,0,83,151,4.505376
359748,359748,This is a trade with OIL-SPEC-HEDGE-NG (John L...,NaN,Trade with John Lavorato,NaN,john.zufferli@enron.com,kori.loibl@enron.com,2001-11-28 21:30:11,NaN,NaN,0,21.0,2.0,0,0,48,87,3.293333
359749,359749,Some of my position is with the Alberta Term b...,NaN,Gas Hedges,NaN,john.zufferli@enron.com,john.lavorato@enron.com,2001-11-28 20:47:48,NaN,NaN,0,20.0,2.0,0,0,23,33,4.000000


In [15]:
import nltk
from nltk.corpus import stopwords

# Download the stop words list if you haven't already
nltk.download('stopwords', quiet=True)

# Create a fast-lookup set of English stop words
STOP_WORDS = set(stopwords.words('english'))

def stop_word_ratio(text):
    if pd.isna(text) or not str(text).strip():
        return 0.0
    
    # Extract clean, lowercase words
    words = re.findall(r'\b\w+\b', str(text).lower())
    
    if not words:
        return 0.0
    
    # Count how many words are in the stop word set
    stop_count = sum(1 for word in words if word in STOP_WORDS)
    
    # Calculate the percentage/ratio
    return stop_count / len(words)

# Apply it to your DataFrame
master_enron['stop_word_ratio'] = master_enron['EmailSend'].apply(stop_word_ratio)

# Preview your updated dataframe
master_enron[['EmailSend', 'EmailSend_Token_Count', 'stop_word_ratio']].head()

,EmailSend,EmailSend_Token_Count,stop_word_ratio
0,"Nikki, Thanks for the note and I hope your doi...",504,0.547758
1,Jim Lokay Sales Representative British Parts I...,11,0.000000
2,Congratulations! Have a good sleep.,5,0.400000
3,Test,1,0.000000
4,We are dropping a lot of marketers. It would b...,16,0.500000
